In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_experimental.text_splitter import SemanticChunker
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_ollama import ChatOllama, OllamaEmbeddings
from dotenv import load_dotenv

load_dotenv()

C:\Users\yashit\AppData\Local\Temp\ipykernel_31216\380218510.py:3: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker
c:\YASHIT\AI\LangChain_Models\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [5]:
import logging

# Turn on multi-query logging to see what is happening in the background
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [6]:
model = ChatOllama(model = 'gemma4:e4b', temperature=0.2, num_ctx=4000)


In [7]:
youtube_video_id = "PHe0bXAIuk0"

ytt_api = YouTubeTranscriptApi()

fetched_transcript = ytt_api.fetch(
    youtube_video_id, 
    languages=["en"]
)

# print(fetched_transcript.snippets)
transcript = " ".join(item.text for item in fetched_transcript.snippets)

print(transcript)

  How the economic machine works, in 30 minutes. The economy works like a simple machine. But many people don't understand it — or they don't agree on how it works — and this has led to a lot of needless economic suffering. I feel a deep sense of responsibility to share my simple but practical economic template. Though it's unconventional, it has helped me to anticipate 
and sidestep the global financial crisis, and has worked well for me for over 30 years. Let's begin. Though the economy might seem complex, 
it works in a simple, mechanical way. It's made up of a few simple parts and a lot of simple transactions that are repeated over and over again a zillion times. These transactions are above all else driven by human nature, and they create 3 main forces that drive the economy. Number 1: Productivity growth Number 2: The Short term debt cycle and Number 3: The Long term debt cycle We'll look at these three forces
and how laying them on top of each other creates a good template for t

In [8]:
embedder = OllamaEmbeddings(model = 'nomic-embed-text:v1.5')

In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)
docs = splitter.split_text(transcript,)
print(docs)
print(len(docs))

["How the economic machine works, in 30 minutes. The economy works like a simple machine. But many people don't understand it — or they don't agree on how it works — and this has led to a lot of needless economic suffering. I feel a deep sense of responsibility to share my simple but practical economic template. Though it's unconventional, it has helped me to anticipate \nand sidestep the global financial crisis, and has worked well for me for over 30 years. Let's begin. Though the economy might seem complex, \nit works in a simple, mechanical way. It's made up of a few simple parts and a lot of simple transactions that are repeated over and over again a zillion times. These transactions are above all else driven by human nature, and they create 3 main forces that drive the economy. Number 1: Productivity growth Number 2: The Short term debt cycle and Number 3: The Long term debt cycle We'll look at these three forces", "and how laying them on top of each other creates a good template 

In [10]:
vector_store = Chroma.from_texts(
    texts=docs,
    embedding=embedder,
    collection_name='transcript'
)

In [11]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['fb5abe1e-f281-4ec8-8484-c1c20c632073',
  'fed431d5-fdc3-4db9-b1ab-ea9fa0e0c22f',
  '04b6deeb-fb6c-44bd-8864-93034981897d',
  'b722f48b-2ba0-4556-b466-e73975bc6771',
  '75390ba9-d434-4d44-8287-165acc1baaed',
  'bb296ec3-078f-44fe-9d86-d02fa0d6545c',
  '493ae4fc-f89f-4117-8ba3-32d05aedfad5',
  'a73bd46e-839b-419d-9de5-618b5cebbcc8',
  'fe337fae-bf10-4a4b-8c12-f3c5e7ef48ed',
  '8bd14c18-531a-4780-8c85-c25ae011a77d',
  'de171c54-15cb-4e68-9da7-4459b07122ae',
  '22652641-d80e-4f87-a884-b9e4eec3138b',
  '6623e2bf-41f9-4651-9766-8cc993de9671',
  '47aa739b-cb41-46e3-ac71-32b72c6006ed',
  'd72d758c-55cd-40e3-99b6-8c3ff1cda032',
  'bcf49efe-145c-4599-85c3-38741f4bb042',
  'efd51403-cc01-4f90-b1d1-1816460759f7',
  '853d61e4-5148-4785-91a6-7beb2cd13925',
  '26c45494-d0c9-4d18-a49f-884fc6adc996',
  'ecf84f81-9a69-42ac-a0a1-603366b751a1',
  'a7023241-fc71-4b48-95a3-0a7661cb4045',
  'e9bcd04a-f658-4511-b6a2-f89d39867750',
  'a5b398f2-0662-4715-b85c-29d27677eeb7',
  '666b1a2a-aab7-440f-8c78-

In [12]:
retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k':5, 'lambda_mult':0.5}
)


In [13]:
multi_query_prompt = PromptTemplate(
    input_variables=["question"],
    template="""You are an AI language model assistant. Your task is to generate 3 different versions of the given user 
    question to retrieve relevant documents from a vector database.Provide these alternative questions separated by newlines. 
    DO NOT include any conversational text, introductions, or numbered lists. ONLY output the text of the questions.

    Original question: {question}"""
)

# sub_query_model = ChatOllama(model = 'gemma4:e4b')
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=model,
    prompt=multi_query_prompt
)

In [14]:
# model = ChatOllama(model = 'gemma4:e4b', temperature=0.2, max_tokens=4000)
compressor = LLMChainExtractor.from_llm(llm=model)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=multi_query_retriever
)

In [15]:
# model = ChatOllama(model = 'gemma4:e4b', temperature=0.2, max_tokens=4000)

In [16]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [17]:
def mergeDocuments(docs):
    global model
    print(docs)
    res = '\n\n'.join(item.page_content for item in docs)
    print(model.get_num_tokens(res))
    print(len(docs))
    return res

In [18]:
parser = StrOutputParser()

In [19]:
parallel_chain = RunnableParallel(
    {"context" : multi_query_retriever | RunnableLambda(mergeDocuments) | parser,
    "question" : RunnablePassthrough()
    }
)

In [20]:
query = "what is economic cycle and explain all the phases of economic cycle in detail"
final_chain = parallel_chain | prompt | model

# 2. Invoke the chain to get the raw AIMessage
response = final_chain.invoke(query)

# 3. Access provider-reported token usage
usage = response.usage_metadata
print("Input (Prompt) Tokens: ", usage["input_tokens"])
print("Output Tokens:         ", usage["output_tokens"])
print("Total Tokens:          ", usage["total_tokens"])

# 4. Access the actual generated text
answer_text = response.content
print("\nAnswer:\n", answer_text)
# print(res)

[Document(metadata={}, page_content="and how laying them on top of each other creates a good template for tracking economic movements and figuring out what's happening now. Let's start with the simplest part of the economy: Transactions. An economy is simply the sum \nof the transactions that make it up and a transaction is a very simple thing. You make transactions all the time. Every time you buy something \nyou create a transaction. Each transaction consists of a buyer exchanging money or credit with a seller for goods, \nservices or financial assets. Credit spends just like money, so adding together the money spent \nand the amount of credit spent, you can know the total spending. The total amount of spending \ndrives the economy. If you divide the amount spent by the quantity sold, you get the price. And that's it.  That's a transaction. It is the building block \nof the economic machine. All cycles and all forces \nin an economy are driven by transactions. So, if we can \nunderst

c:\YASHIT\AI\LangChain_Models\venv\Lib\site-packages\langchain_core\language_models\base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


1764
10
Input (Prompt) Tokens:  1870
Output Tokens:          1247
Total Tokens:           3117

Answer:
 Economic cycles are driven by transactions and are primarily influenced by the amount of credit. The economy is driven by three main forces: Productivity growth, the Short term debt cycle, and the Long term debt cycle.

**The Short Term Debt Cycle (typically lasts 5–8 years):**

1.  **Expansion:** Economic activity increases. Spending continues to increase, and prices start to rise (inflation). This increase in spending is fueled by credit, which can be created instantly. When spending and incomes grow faster than the production of goods, prices rise.
2.  **Recession:** Seeing prices rise, the Central Bank raises interest rates. With higher interest rates, fewer people can afford to borrow money, and the cost of existing debts rises. When credit isn't easily available, there is a recession.

**The Long Term Debt Cycle (takes about 75 to 100 years):**

1.  **Long Term Debt Peak/Delev